# Extracción de métricas acústicas y lingüísticas

Este notebook está basado en el enfoque descrito en el artículo [PMC9056005](https://pmc.ncbi.nlm.nih.gov/articles/PMC9056005/), adaptado para analizar chunks de audio y sus correspondientes transcripciones en un contexto de clasificación de afasia.

## Métricas Extraídas

Las siguientes métricas se generan a partir de los audios y transcripciones procesados:

### **Métricas Acústicas (OpenSMILE eGeMAPS)**
- `F0semitoneFrom27.5Hz_sma3nz_amean`
- `F0semitoneFrom27.5Hz_sma3nz_stddevNorm`
- `F0semitoneFrom27.5Hz_sma3nz_percentile20.0`
- `F0semitoneFrom27.5Hz_sma3nz_percentile50.0`
- `F0semitoneFrom27.5Hz_sma3nz_percentile80.0`
- `F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2`
- `F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope`
- `F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope`
- `F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope`
- `F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope`
- `loudness_sma3_amean`
- `loudness_sma3_stddevNorm`
- `spectralFlux_sma3_amean`
- `spectralFlux_sma3_stddevNorm`
- `mfcc1_sma3_amean` a `mfcc4_sma3_stddevNorm`
- `jitterLocal_sma3nz_amean`
- `jitterLocal_sma3nz_stddevNorm`
- `shimmerLocaldB_sma3nz_amean`
- `shimmerLocaldB_sma3nz_stddevNorm`
- `HNRdBACF_sma3nz_amean`
- `HNRdBACF_sma3nz_stddevNorm`
- `alphaRatioV_sma3nz_amean`
- `alphaRatioV_sma3nz_stddevNorm`
- `hammarbergIndexV_sma3nz_amean`
- `hammarbergIndexV_sma3nz_stddevNorm`

### **Métricas Lingüísticas**
- Número total de palabras (`num_palabras`).
- Número de palabras únicas (`num_palabras_unicas`).
- Diversidad léxica (`diversidad_lexica`): proporción de palabras únicas respecto al total.
- Embeddings de BERT para las transcripciones.

### **Métricas Derivadas (Librosa)**
- `mfcc1_mean` a `mfcc13_mean`: Medias de los coeficientes MFCC.
- `mfcc1_stddev` a `mfcc13_stddev`: Desviaciones estándar de los coeficientes MFCC.

Estas métricas se combinan con las columnas originales del dataset para formar un único conjunto de datos listo para la modelización.

In [1]:
import librosa
import json
import numpy as np
import pandas as pd
from opensmile import Smile,FeatureSet, FeatureLevel
from transformers import BertTokenizer, BertModel
import spacy

In [2]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)   

In [3]:
# Inicialización de librerías necesarias
nlp = spacy.load("es_core_news_sm")  # Para métricas lingüísticas
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")  # BERT embeddings
model = BertModel.from_pretrained("bert-base-uncased")

# Inicializar OpenSMILE para características acústicas específicas
smile_egemaps = Smile(feature_set=FeatureSet.eGeMAPSv02, feature_level=FeatureLevel.Functionals)

# Función para extraer características acústicas con librosa
def extraer_mfcc(y, sr):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta_mfcc = librosa.feature.delta(mfccs)
    delta2_mfcc = librosa.feature.delta(mfccs, order=2)
    
    # Calcular estadísticas resumen
    features = {
        f"mfcc{i+1}_mean": np.mean(mfcc) for i, mfcc in enumerate(mfccs)
    }
    features.update({
        f"mfcc{i+1}_stddev": np.std(mfcc) for i, mfcc in enumerate(mfccs)
    })
    return features

# Función para extraer características lingüísticas
def extraer_linguisticas(transcripcion):
    # Procesamiento de texto con spaCy
    doc = nlp(transcripcion)
    num_palabras = len(doc)
    num_palabras_unicas = len(set([token.text for token in doc]))
    diversidad_lexica = num_palabras_unicas / num_palabras if num_palabras > 0 else 0
    
    # Embeddings de BERT
    inputs = tokenizer(transcripcion, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    bert_sent_embedding = outputs.last_hidden_state.mean(dim=1).detach().numpy().flatten()
    
    return {
        "num_palabras": num_palabras,
        "num_palabras_unicas": num_palabras_unicas,
        "diversidad_lexica": diversidad_lexica,
        "bert_embedding": bert_sent_embedding
    }

# Función principal para extraer TODAS las métricas para un chunk
def extraer_metricas_chunk(ruta_audio, transcripcion):
    try:
        # Cargar audio
        y, sr = librosa.load(ruta_audio, sr=None)

        # Características acústicas con librosa (MFCCs)
        mfcc_features = extraer_mfcc(y, sr)

        # Características acústicas con openSMILE
        egemaps_features = smile_egemaps.process_file(ruta_audio).to_dict(orient='records')[0]

        # Características lingüísticas
        linguistics = extraer_linguisticas(transcripcion)

        # Fusionar todas las métricas
        all_features = {
            **mfcc_features,
            **egemaps_features,
            **linguistics  # Incluye características lingüísticas
        }
        return all_features
    except Exception as e:
        print(f"Error procesando el audio {ruta_audio}: {e}")
        return None

In [4]:
path_data = '/Users/monicaromero/PycharmProjects/afasia_cat/notebooks_202412/data/'
df = pd.read_csv(path_data + 'df_transcrip_chunk_info.csv', encoding='utf-8')

resultados = []

# Procesar cada fila del dataset
for idx, row in df.iterrows():
    ruta_audio = row['name_chunk_audio_path']
    transcripcion = row['Marca']  # Columna con las transcripciones

    # Extraer métricas para cada chunk
    metricas = extraer_metricas_chunk(ruta_audio, transcripcion)

    if metricas:
        # Serializar bert_embedding si está presente
        if 'bert_embedding' in metricas:
            # Convertir ndarray a lista antes de serializar
            if isinstance(metricas['bert_embedding'], np.ndarray):
                metricas['bert_embedding'] = metricas['bert_embedding'].tolist()
            metricas['bert_embedding'] = json.dumps(metricas['bert_embedding'])

        resultado = row.to_dict()  # Copia todas las columnas originales
        resultado.update(metricas)  # Añade las nuevas métricas
        resultados.append(resultado)

# Convertir resultados a DataFrame
df_resultados = pd.DataFrame(resultados)

In [5]:
df_resultados.head()

,Inicio,Fin,Marca,Transcrip_name,Duración,name_chunk_audio,name_chunk_audio_path,CIP,NumId,Gènere,TipusAfàsia,LLengWAB,Edat,Grup,QA,Fluente/No Fluente,num_palabras,num_palabras_unicas,promedio_palabras_por_frase,num_ininteligibles,palabras_por_minuto,palabras_por_segundo,mfcc1_mean,mfcc2_mean,mfcc3_mean,mfcc4_mean,mfcc5_mean,mfcc6_mean,mfcc7_mean,mfcc8_mean,mfcc9_mean,mfcc10_mean,mfcc11_mean,mfcc12_mean,mfcc13_mean,mfcc1_stddev,mfcc2_stddev,mfcc3_stddev,mfcc4_stddev,mfcc5_stddev,mfcc6_stddev,mfcc7_stddev,mfcc8_stddev,mfcc9_stddev,mfcc10_stddev,mfcc11_stddev,mfcc12_stddev,mfcc13_stddev,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope,F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope,loudness_sma3_amean,loudness_sma3_stddevNorm,loudness_sma3_percentile20.0,loudness_sma3_percentile50.0,loudness_sma3_percentile80.0,loudness_sma3_pctlrange0-2,loudness_sma3_meanRisingSlope,loudness_sma3_stddevRisingSlope,loudness_sma3_meanFallingSlope,loudness_sma3_stddevFallingSlope,spectralFlux_sma3_amean,spectralFlux_sma3_stddevNorm,mfcc1_sma3_amean,mfcc1_sma3_stddevNorm,mfcc2_sma3_amean,mfcc2_sma3_stddevNorm,mfcc3_sma3_amean,mfcc3_sma3_stddevNorm,mfcc4_sma3_amean,mfcc4_sma3_stddevNorm,jitterLocal_sma3nz_amean,jitterLocal_sma3nz_stddevNorm,shimmerLocaldB_sma3nz_amean,shimmerLocaldB_sma3nz_stddevNorm,HNRdBACF_sma3nz_amean,HNRdBACF_sma3nz_stddevNorm,logRelF0-H1-H2_sma3nz_amean,logRelF0-H1-H2_sma3nz_stddevNorm,logRelF0-H1-A3_sma3nz_amean,logRelF0-H1-A3_sma3nz_stddevNorm,F1frequency_sma3nz_amean,F1frequency_sma3nz_stddevNorm,F1bandwidth_sma3nz_amean,F1bandwidth_sma3nz_stddevNorm,F1amplitudeLogRelF0_sma3nz_amean,F1amplitudeLogRelF0_sma3nz_stddevNorm,F2frequency_sma3nz_amean,F2frequency_sma3nz_stddevNorm,F2bandwidth_sma3nz_amean,F2bandwidth_sma3nz_stddevNorm,F2amplitudeLogRelF0_sma3nz_amean,F2amplitudeLogRelF0_sma3nz_stddevNorm,F3frequency_sma3nz_amean,F3frequency_sma3nz_stddevNorm,F3bandwidth_sma3nz_amean,F3bandwidth_sma3nz_stddevNorm,F3amplitudeLogRelF0_sma3nz_amean,F3amplitudeLogRelF0_sma3nz_stddevNorm,alphaRatioV_sma3nz_amean,alphaRatioV_sma3nz_stddevNorm,hammarbergIndexV_sma3nz_amean,hammarbergIndexV_sma3nz_stddevNorm,slopeV0-500_sma3nz_amean,slopeV0-500_sma3nz_stddevNorm,slopeV500-1500_sma3nz_amean,slopeV500-1500_sma3nz_stddevNorm,spectralFluxV_sma3nz_amean,spectralFluxV_sma3nz_stddevNorm,mfcc1V_sma3nz_amean,mfcc1V_sma3nz_stddevNorm,mfcc2V_sma3nz_amean,mfcc2V_sma3nz_stddevNorm,mfcc3V_sma3nz_amean,mfcc3V_sma3nz_stddevNorm,mfcc4V_sma3nz_amean,mfcc4V_sma3nz_stddevNorm,alphaRatioUV_sma3nz_amean,hammarbergIndexUV_sma3nz_amean,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp,diversidad_lexica,bert_embedding
0,0.00000,2.32800,IL,02_008_CAT_Conversacion,2.32800,02_008_CAT_Conversacion_0.00000_2.32800.wav,/Users/monicaromero/PycharmProjects/afasia_cat...,02_008,8,1,5,2,71,2,82.6,Fluente,1,1,1.0,1,25.773196,0.429553,-468.575775,80.942200,24.115252,30.720064,-3.104674,12.531712,-19.845417,-0.400819,-17.247997,-0.449294,-4.693883,-0.836542,-9.179538,25.112869,24.200817,13.256679,9.368532,10.391405,9.894167,9.756401,9.506847,7.677062,6.049176,7.807673,7.396701,6.960604,21.080507,0.092358,19.728111,20.638460,21.581902,1.853790,31.598660,15.512374,22.600626,23.475050,0.237135,0.350971,0.177719,0.219633,0.266333,0.088615,2.407893,2.324636,1.064285,0.718014,0.072825,0.906789,14.468367,0.560155,12.692016,0.494521,1.987319,4.188074,-8.812879,-1.005138,0.029396,1.469401,1.254249,0.616065,0.693947,5.214302,-6.523772,-0.986962,12.733748,0.568

In [6]:
print(list(df_resultados.columns))

['Inicio', 'Fin', 'Marca', 'Transcrip_name', 'Duración', 'name_chunk_audio', 'name_chunk_audio_path', 'CIP', 'NumId', 'Gènere', 'TipusAfàsia', 'LLengWAB', 'Edat', 'Grup', 'QA', 'Fluente/No Fluente', 'num_palabras', 'num_palabras_unicas', 'promedio_palabras_por_frase', 'num_ininteligibles', 'palabras_por_minuto', 'palabras_por_segundo', 'mfcc1_mean', 'mfcc2_mean', 'mfcc3_mean', 'mfcc4_mean', 'mfcc5_mean', 'mfcc6_mean', 'mfcc7_mean', 'mfcc8_mean', 'mfcc9_mean', 'mfcc10_mean', 'mfcc11_mean', 'mfcc12_mean', 'mfcc13_mean', 'mfcc1_stddev', 'mfcc2_stddev', 'mfcc3_stddev', 'mfcc4_stddev', 'mfcc5_stddev', 'mfcc6_stddev', 'mfcc7_stddev', 'mfcc8_stddev', 'mfcc9_stddev', 'mfcc10_stddev', 'mfcc11_stddev', 'mfcc12_stddev', 'mfcc13_stddev', 'F0semitoneFrom27.5Hz_sma3nz_amean', 'F0semitoneFrom27.5Hz_sma3nz_stddevNorm', 'F0semitoneFrom27.5Hz_sma3nz_percentile20.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile50.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile80.0', 'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2', 'F0

In [7]:
ruta_base = '/Users/monicaromero/PycharmProjects/afasia_cat/notebooks_202412/data/'
df_resultados.to_csv(ruta_base + 'df_transcrip_audio_metrics.csv', index=False, encoding='utf-8')

# Aphasiabank

In [8]:
df_aphbank = pd.read_csv('/Users/monicaromero/PycharmProjects/afasia_cat/notebooks_202412/data/df_aphbank_transcrip_chunk_info.csv')
df_aphbank.columns

Index(['Inicio', 'Fin', 'Marca', 'sex', 'Edat', 'Transcrip_name', 'QA',
       'aphasia_type', 'Grup', 'fluency_speech', 'name_chunk_audio',
       'Duración', 'num_words', 'Gènere', 'TipusAfàsia', 'Fluente/No Fluente',
       'LLengWAB', 'CIP', 'name_chunk_audio_path', 'num_palabras',
       'num_palabras_unicas', 'promedio_palabras_por_frase',
       'num_ininteligibles', 'palabras_por_minuto', 'palabras_por_segundo'],
      dtype='object')

In [9]:
resultados_aphbank = []

# Procesar cada fila del dataset
for idx, row in df_aphbank.iterrows():
    ruta_audio = row['name_chunk_audio_path']
    transcripcion = row['Marca']  # Columna con las transcripciones

    # Extraer métricas para cada chunk
    metricas = extraer_metricas_chunk(ruta_audio, transcripcion)

    if metricas:
        # Serializar bert_embedding si está presente
        if 'bert_embedding' in metricas:
            # Convertir ndarray a lista antes de serializar
            if isinstance(metricas['bert_embedding'], np.ndarray):
                metricas['bert_embedding'] = metricas['bert_embedding'].tolist()
            metricas['bert_embedding'] = json.dumps(metricas['bert_embedding'])

        resultado = row.to_dict()  # Copia todas las columnas originales
        resultado.update(metricas)  # Añade las nuevas métricas
        resultados_aphbank.append(resultado)

# Convertir resultados a DataFrame
df_aphbank_resultados = pd.DataFrame(resultados_aphbank)

Error procesando el audio /Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU02a_315.55_0.25.wav: when mode='interp', width=9 cannot exceed data.shape[axis]=8
Error procesando el audio /Users/monicaromero/PycharmProjects/afasia_cat/aphasiabank_es/Audios_ES/TCU10a_844.931_0.237.wav: when mode='interp', width=9 cannot exceed data.shape[axis]=8


In [13]:
ruta_base = '/Users/monicaromero/PycharmProjects/afasia_cat/notebooks_202412/data/'
df_aphbank_resultados.to_csv(ruta_base + 'df_aphasiabank_metrics.csv', index=False, encoding='utf-8')

In [14]:
df_aphbank_resultados['CIP'].unique()

array(['TCU06a', 'TCU04a', 'TCU02a', 'TCU10a'], dtype=object)